In [ ]:
# Cell 1 — Installs
!pip install ultralytics kornia --quiet

In [ ]:
# Cell 2 — Imports
import cv2
import random
import numpy as np
import torch
import torch.nn.functional as F
import kornia
import kornia.augmentation as K
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from ultralytics import YOLO

In [ ]:
# Cell 3 — Config
# ── Training ──────────────────────────────────────────
PATCH_H       = 200      # patch height in pixels
PATCH_W       = 150      # patch width in pixels
PATCH_SCALE   = 0.3      # fraction of detected person width used for patch
SHIRT_TOP     = 0.30     # top of shirt region as fraction of person bbox height
SHIRT_BOT     = 0.70     # bottom of shirt region as fraction of person bbox height
EPSILON       = 0.15     # max pixel delta from initialization (0–1 range)
LR            = 0.01     # Adam learning rate
NUM_STEPS     = 200      # total training steps (use 30 for debug run)
EOT_N         = 8        # EoT transforms per step
TOP_K         = 50       # top anchors to target in confidence loss
CONF_LOSS_W   = 0.6      # weight for Top-K confidence loss
SEG_LOSS_W    = 0.4      # weight for mask suppression loss

# ── Input ──────────────────────────────────────────────
# Upload one image to Colab and set this path.
# Any image with a clearly visible person works (e.g., someone walking).
IMAGE_PATH    = "person.jpg"

# ── Model ──────────────────────────────────────────────
YOLO_MODEL    = "yolov8n-seg"   # nano for speed; swap yolov8x-seg for eval
DEVICE        = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

In [ ]:
# Cell 4 — Load YOLOv8-seg, freeze weights
yolo = YOLO(YOLO_MODEL)
torch_model = yolo.model.to(DEVICE)
torch_model.eval()

# Freeze all weights — only patch pixels will be updated
for p in torch_model.parameters():
    p.requires_grad_(False)

print(f"Model loaded: {YOLO_MODEL} | Parameters frozen: {sum(p.numel() for p in torch_model.parameters()):,}")

In [ ]:
# Cell 5 — Raw forward pass helper
def yolo_raw_forward(img_tensor):
    """
    img_tensor: [1, 3, 640, 640] float32 on DEVICE, values in [0, 1]
    Returns dict:
        'preds': [1, 116, 8400]  — 4 box + 80 class logits + 32 mask coeffs per anchor
        'proto': [1, 32, 160, 160] — mask prototype basis
    """
    out = torch_model(img_tensor)
    return {'preds': out[0], 'proto': out[1][0]}


def img_to_tensor(img_np):
    """uint8 HxWx3 numpy → [1, 3, 640, 640] float32 tensor on DEVICE"""
    img = cv2.resize(img_np, (640, 640))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    t = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0
    return t.unsqueeze(0).to(DEVICE)

In [ ]:
# Cell 5b — Sanity check (delete after verification)
# Quick sanity check — run in its own cell temporarily, delete after
test_img = np.zeros((480, 640, 3), dtype=np.uint8)
raw = yolo_raw_forward(img_to_tensor(test_img))
print("preds shape:", raw['preds'].shape)   # expect [1, 116, 8400]
print("proto shape:", raw['proto'].shape)   # expect [1, 32, 160, 160]